# Correlation Study: AI Vibrancy, Economic Freedom, and Regulatory Framing

This notebook tests the relationship between Stanford AI Vibrancy, Heritage Index of Economic Freedom (IEF), and sentence-level AI regulatory framing for G7 + China.

The default framing measure is the corrected DeepSeek sentence-level output calibrated with Daria + Stephen human consensus labels, excluding human-gold `Unsure` labels. Older Qwen or semantic outputs can be added as robustness checks, but the analysis below is built around the new corrected framing file.


## Hypotheses

**H1 — AI Vibrancy and Innovation-Oriented Rhetoric**  
Countries with higher Stanford AI Vibrancy scores will exhibit more innovation-oriented language in their national AI policy documents.

**H2 — AI Vibrancy and Risk-Oriented Rhetoric**  
Countries with lower Stanford AI Vibrancy scores will exhibit more risk-oriented language in their national AI policy documents.

**H3 — Economic Freedom and Innovation-Oriented Regulatory Framing**  
Countries with higher economic freedom scores will exhibit more innovation-oriented AI regulatory framing in their policy documents.

**H4 — Risk-Oriented Regulatory Framing and Economic Freedom**  
Countries whose AI regulatory discourse is more negative and risk-oriented will exhibit lower economic freedom scores.

Because this is an eight-country cross-sectional study, all inferential statistics should be interpreted as exploratory and directional rather than confirmatory.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'data').exists():
    repo_root = repo_root.parent

outputs = repo_root / 'outputs'
outputs.mkdir(parents=True, exist_ok=True)

G7_CHINA_COUNTRIES = [
    'United States',
    'China',
    'United Kingdom',
    'France',
    'Germany',
    'Italy',
    'Japan',
    'Canada',
]

# Main analysis source: corrected DeepSeek labels calibrated by Daria + Stephen human consensus labels.
FRAMING_SOURCE = 'deepseek_corrected'
FRAMING_TAG = 'daria_stephen_no_unsure_full'
EPSILON = 0.5

repo_root


## Load and Merge Data

In [ ]:
def load_vibrancy():
    path = repo_root / 'data' / 'AI vibrancy tool screen shot' / 'ai_country_scores.csv'
    df = pd.read_csv(path)
    return df.rename(columns={'Total Score': 'AI Vibrancy Score'})


def load_latest_ief():
    path = repo_root / 'data' / 'number' / 'ief_g7_china_panel.csv'
    df = pd.read_csv(path)
    latest_year = int(df['Index Year'].max())
    latest = df[df['Index Year'] == latest_year].copy()
    latest = latest.rename(columns={'Overall Score': 'IEF Overall Score'})
    return latest[['Country', 'Index Year', 'IEF Overall Score']]


def load_framing(source='deepseek_corrected'):
    if source != 'deepseek_corrected':
        raise ValueError("This notebook is currently configured for source='deepseek_corrected'.")

    path = outputs / f'deepseek_ai_framing_summary_corrected_{FRAMING_TAG}.csv'
    df = pd.read_csv(path).rename(columns={
        'country': 'Country',
        'Innovation-oriented': 'Innovation-oriented Sentences',
        'Mixed': 'Mixed Sentences',
        'Neutral': 'Neutral Sentences',
        'Risk-oriented': 'Risk-oriented Sentences',
        'Unsure': 'Unsure Sentences',
        'total_sentences': 'Total Sentences',
        'net_framing_score': 'Net Innovation Framing',
        'mean_framing_score': 'Mean Framing Score',
        'innovation_to_risk_ratio': 'Innovation to Risk Ratio',
    })

    df['Framing Source'] = 'DeepSeek corrected, Daria + Stephen no-unsure full'
    df['Innovation Share'] = df['Innovation-oriented Sentences'] / df['Total Sentences']
    df['Risk Share'] = df['Risk-oriented Sentences'] / df['Total Sentences']
    df['Mixed Share'] = df['Mixed Sentences'] / df['Total Sentences']
    df['Neutral Share'] = df['Neutral Sentences'] / df['Total Sentences']
    df['Innovation per 100 Sentences'] = df['Innovation Share'] * 100
    df['Risk per 100 Sentences'] = df['Risk Share'] * 100
    df['Log Innovation to Risk Ratio'] = np.log(df['Innovation to Risk Ratio'])

    return df[[
        'Country', 'Framing Source', 'Total Sentences',
        'Innovation-oriented Sentences', 'Risk-oriented Sentences', 'Mixed Sentences',
        'Neutral Sentences', 'Unsure Sentences',
        'Innovation Share', 'Risk Share', 'Mixed Share', 'Neutral Share',
        'Innovation per 100 Sentences', 'Risk per 100 Sentences',
        'Innovation to Risk Ratio', 'Log Innovation to Risk Ratio',
        'Net Innovation Framing', 'Mean Framing Score',
    ]]


vibrancy = load_vibrancy()
ief = load_latest_ief()
framing = load_framing(FRAMING_SOURCE)

study = (
    pd.DataFrame({'Country': G7_CHINA_COUNTRIES})
    .merge(vibrancy, on='Country', how='left')
    .merge(ief, on='Country', how='left')
    .merge(framing, on='Country', how='left')
)

excluded_framing_cases = sorted(set(framing['Country']) - set(G7_CHINA_COUNTRIES))

study_path = outputs / f'correlation_study_dataset_{FRAMING_SOURCE}_{FRAMING_TAG}.csv'
study.to_csv(study_path, index=False)
print(f'Saved: {study_path}')
print(f'Excluded from G7 + China correlation sample because AI Vibrancy/IEF are unavailable here: {excluded_framing_cases}')
study


## Correlation Tests

In [ ]:
def correlation_row(df, x, y, hypothesis, expected_direction):
    clean = df[[x, y]].dropna()
    pearson_r, pearson_p = stats.pearsonr(clean[x], clean[y])
    spearman_r, spearman_p = stats.spearmanr(clean[x], clean[y])
    directional_support = (
        pearson_r > 0 if expected_direction == 'positive' else pearson_r < 0
    )
    return {
        'Hypothesis': hypothesis,
        'X': x,
        'Y': y,
        'Expected Direction': expected_direction,
        'N': len(clean),
        'Pearson r': round(float(pearson_r), 3),
        'Pearson p': round(float(pearson_p), 4),
        'Spearman rho': round(float(spearman_r), 3),
        'Spearman p': round(float(spearman_p), 4),
        'Directional Support': bool(directional_support),
    }


corr_rows = [
    correlation_row(study, 'AI Vibrancy Score', 'Innovation Share', 'H1: Vibrancy -> innovation framing', 'positive'),
    correlation_row(study, 'AI Vibrancy Score', 'Risk Share', 'H2: Lower vibrancy -> risk framing', 'negative'),
    correlation_row(study, 'IEF Overall Score', 'Innovation Share', 'H3: IEF -> innovation framing', 'positive'),
    correlation_row(study, 'Risk Share', 'IEF Overall Score', 'H4: Risk framing -> lower IEF', 'negative'),
    correlation_row(study, 'AI Vibrancy Score', 'Log Innovation to Risk Ratio', 'Ratio robustness: Vibrancy -> innovation/risk balance', 'positive'),
    correlation_row(study, 'IEF Overall Score', 'Log Innovation to Risk Ratio', 'Ratio robustness: IEF -> innovation/risk balance', 'positive'),
    correlation_row(study, 'AI Vibrancy Score', 'IEF Overall Score', 'Background association: Vibrancy and IEF', 'positive'),
]
correlations = pd.DataFrame(corr_rows)

corr_path = outputs / f'correlation_study_correlations_{FRAMING_SOURCE}_{FRAMING_TAG}.csv'
correlations.to_csv(corr_path, index=False)
print(f'Saved: {corr_path}')
correlations


## Outlier Sensitivity Check

The main analysis keeps all eight G7 + China cases. As a robustness and interpretation check, this section evaluates two theoretically salient outliers. These exclusions are not used to replace the full-sample results; they are used to test whether substantively unusual cases are driving the observed correlations.

- **United States for H1/H2**: the United States is by far the highest-AI-vibrancy case in the sample, but its corrected framing is comparatively risk-conscious. It combines very high AI ecosystem capacity with relatively low innovation share and high risk share. This makes it a high-leverage case for tests linking AI Vibrancy to innovation- or risk-oriented rhetoric.
- **China for H3/H4**: China has the lowest IEF score in the sample but one of the strongest innovation-oriented framing profiles. It is therefore a theoretically important state-led innovation case that can strongly influence tests linking economic freedom to innovation- or risk-oriented rhetoric.

With only seven observations after each exclusion, effect direction and magnitude are more informative than p-values. The goal is to identify sensitivity to influential cases, not to choose a preferred specification after seeing the results.


In [ ]:
def outlier_sensitivity_row(df, x, y, hypothesis, excluded_country, expected_direction):
    full = correlation_row(df, x, y, f'{hypothesis} full sample', expected_direction)
    subset = df[df['Country'] != excluded_country].copy()
    excl = correlation_row(subset, x, y, f'{hypothesis} excluding {excluded_country}', expected_direction)
    return {
        'Hypothesis': hypothesis,
        'Excluded Country': excluded_country,
        'X': x,
        'Y': y,
        'Expected Direction': expected_direction,
        'Full N': full['N'],
        'Full Pearson r': full['Pearson r'],
        'Full Pearson p': full['Pearson p'],
        'Full Spearman rho': full['Spearman rho'],
        'Full Spearman p': full['Spearman p'],
        'Full Directional Support': full['Directional Support'],
        'Sensitivity N': excl['N'],
        'Sensitivity Pearson r': excl['Pearson r'],
        'Sensitivity Pearson p': excl['Pearson p'],
        'Sensitivity Spearman rho': excl['Spearman rho'],
        'Sensitivity Spearman p': excl['Spearman p'],
        'Sensitivity Directional Support': excl['Directional Support'],
    }


outlier_sensitivity = pd.DataFrame([
    outlier_sensitivity_row(
        study,
        'AI Vibrancy Score',
        'Innovation Share',
        'H1: Vibrancy -> innovation framing',
        'United States',
        'positive',
    ),
    outlier_sensitivity_row(
        study,
        'AI Vibrancy Score',
        'Risk Share',
        'H2: Lower vibrancy -> risk framing',
        'United States',
        'negative',
    ),
    outlier_sensitivity_row(
        study,
        'IEF Overall Score',
        'Innovation Share',
        'H3: IEF -> innovation framing',
        'China',
        'positive',
    ),
    outlier_sensitivity_row(
        study,
        'Risk Share',
        'IEF Overall Score',
        'H4: Risk framing -> lower IEF',
        'China',
        'negative',
    ),
])

sensitivity_path = outputs / f'correlation_study_outlier_sensitivity_{FRAMING_SOURCE}_{FRAMING_TAG}.csv'
outlier_sensitivity.to_csv(sensitivity_path, index=False)
print(f'Saved: {sensitivity_path}')
outlier_sensitivity


### Outlier Sensitivity Interpretation

The United States is a high-vibrancy but relatively risk-conscious case. Excluding it changes H1 from essentially no relationship to a strong positive association between AI Vibrancy and Innovation Share. H2 also flips into the expected negative direction, but remains weak.

China is a low-economic-freedom but strongly innovation-oriented case. Excluding it weakens the negative H3 relationship, but does not produce support for the expected positive IEF-to-innovation relationship. H4 remains opposite to expectation and becomes stronger by Pearson correlation.

Substantively, the outlier check suggests that the United States and China are not merely statistical inconveniences; they are theoretically important cases. The United States looks like a high-capacity, risk-conscious AI governance case, while China looks like a low-IEF, state-led innovation-framing case. The full sample remains the main specification, and the exclusions should be read as sensitivity diagnostics.


## OLS Models

The regression models mirror H1-H4 and retain a combined model for the innovation-to-risk balance. With only eight countries, these models are descriptive diagnostics rather than confirmatory estimates.


In [ ]:
def fit_ols(df, y_col, x_cols, model_name, keep_cols=None):
    keep_cols = keep_cols or []
    clean = df[[*keep_cols, y_col, *x_cols]].dropna().copy()
    y = clean[y_col].to_numpy(dtype=float)
    x = clean[x_cols].to_numpy(dtype=float)
    x_design = np.column_stack([np.ones(len(clean)), x])
    beta, *_ = np.linalg.lstsq(x_design, y, rcond=None)
    fitted = x_design @ beta
    resid = y - fitted
    n = len(y)
    k = x_design.shape[1]
    sse = float(np.sum(resid ** 2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1 - sse / sst if sst else np.nan
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k) if n > k and not np.isnan(r2) else np.nan
    sigma2 = sse / (n - k) if n > k else np.nan
    cov = sigma2 * np.linalg.pinv(x_design.T @ x_design) if n > k else np.full((k, k), np.nan)
    se = np.sqrt(np.diag(cov))
    t_values = beta / se
    p_values = 2 * stats.t.sf(np.abs(t_values), df=n - k) if n > k else np.full(k, np.nan)

    names = ['Intercept', *x_cols]
    rows = []
    for name, coef, se_i, t_i, p_i in zip(names, beta, se, t_values, p_values):
        rows.append({
            'Model': model_name,
            'Outcome': y_col,
            'Term': name,
            'Coefficient': round(float(coef), 4),
            'Std. Error': round(float(se_i), 4) if np.isfinite(se_i) else np.nan,
            't': round(float(t_i), 3) if np.isfinite(t_i) else np.nan,
            'p': round(float(p_i), 4) if np.isfinite(p_i) else np.nan,
            'N': n,
            'R2': round(float(r2), 3) if np.isfinite(r2) else np.nan,
            'Adj. R2': round(float(adj_r2), 3) if np.isfinite(adj_r2) else np.nan,
        })
    clean['Fitted'] = fitted
    return pd.DataFrame(rows), fitted, clean


m1, _, _ = fit_ols(study, 'Innovation Share', ['AI Vibrancy Score'], 'H1: Innovation share ~ AI vibrancy')
m2, _, _ = fit_ols(study, 'Risk Share', ['AI Vibrancy Score'], 'H2: Risk share ~ AI vibrancy')
m3, _, _ = fit_ols(study, 'Innovation Share', ['IEF Overall Score'], 'H3: Innovation share ~ IEF')
m4, _, _ = fit_ols(study, 'IEF Overall Score', ['Risk Share'], 'H4: IEF ~ risk share')
ratio_vibrancy, _, _ = fit_ols(study, 'Log Innovation to Risk Ratio', ['AI Vibrancy Score'], 'Robustness: Log ratio ~ AI vibrancy')
ratio_ief, _, _ = fit_ols(study, 'Log Innovation to Risk Ratio', ['IEF Overall Score'], 'Robustness: Log ratio ~ IEF')
combined_model, combined_fitted, combined_data = fit_ols(
    study,
    'Log Innovation to Risk Ratio',
    ['AI Vibrancy Score', 'IEF Overall Score'],
    'Combined: Log ratio ~ AI vibrancy + IEF',
    keep_cols=['Country'],
)
models = pd.concat([m1, m2, m3, m4, ratio_vibrancy, ratio_ief, combined_model], ignore_index=True)

model_path = outputs / f'correlation_study_regression_models_{FRAMING_SOURCE}_{FRAMING_TAG}.csv'
models.to_csv(model_path, index=False)
print(f'Saved: {model_path}')
models


## Mediation Diagnostic

This diagnostic follows the mechanism implied by H1, H3, and H4: AI vibrancy may shape regulatory framing, and regulatory framing may be associated with economic freedom.

- X = AI Vibrancy Score
- M = Mean Framing Score, where higher values indicate more innovation-oriented and lower values indicate more risk-oriented rhetoric
- Y = IEF Overall Score

With only eight countries, this is not a confirmatory mediation test. Treat it as a mechanism check.


In [ ]:
def ols_coefficients(df, y_col, x_cols):
    clean = df[[y_col, *x_cols]].dropna().copy()
    y = clean[y_col].to_numpy(dtype=float)
    x = clean[x_cols].to_numpy(dtype=float)
    x_design = np.column_stack([np.ones(len(clean)), x])
    beta, *_ = np.linalg.lstsq(x_design, y, rcond=None)
    return dict(zip(['Intercept', *x_cols], beta))


def coef_for(df, y_col, x_cols, term):
    return float(ols_coefficients(df, y_col, x_cols)[term])


mediation_df = study[['AI Vibrancy Score', 'Mean Framing Score', 'IEF Overall Score']].dropna().copy()
a = coef_for(mediation_df, 'Mean Framing Score', ['AI Vibrancy Score'], 'AI Vibrancy Score')
b = coef_for(mediation_df, 'IEF Overall Score', ['AI Vibrancy Score', 'Mean Framing Score'], 'Mean Framing Score')
c_total = coef_for(mediation_df, 'IEF Overall Score', ['AI Vibrancy Score'], 'AI Vibrancy Score')
c_prime = coef_for(mediation_df, 'IEF Overall Score', ['AI Vibrancy Score', 'Mean Framing Score'], 'AI Vibrancy Score')
indirect = a * b
mediated_share = indirect / c_total if c_total else np.nan

rng = np.random.default_rng(2026)
boot = []
for _ in range(1000):
    sample = mediation_df.sample(len(mediation_df), replace=True, random_state=int(rng.integers(0, 1_000_000_000)))
    try:
        a_i = coef_for(sample, 'Mean Framing Score', ['AI Vibrancy Score'], 'AI Vibrancy Score')
        b_i = coef_for(sample, 'IEF Overall Score', ['AI Vibrancy Score', 'Mean Framing Score'], 'Mean Framing Score')
        indirect_i = a_i * b_i
        if np.isfinite(indirect_i):
            boot.append(indirect_i)
    except Exception:
        continue

ci_low, ci_high = np.percentile(boot, [2.5, 97.5]) if boot else (np.nan, np.nan)
mediation = pd.DataFrame([{
    'N': len(mediation_df),
    'a: Mean framing ~ Vibrancy': round(a, 4),
    'b: IEF ~ Mean framing | Vibrancy': round(b, 4),
    'c total: IEF ~ Vibrancy': round(c_total, 4),
    "c': IEF ~ Vibrancy | Mean framing": round(c_prime, 4),
    'Indirect effect a*b': round(indirect, 4),
    'Bootstrap 95% CI low': round(float(ci_low), 4),
    'Bootstrap 95% CI high': round(float(ci_high), 4),
    'Mediated share': round(float(mediated_share), 4) if np.isfinite(mediated_share) else np.nan,
}])

mediation_path = outputs / f'correlation_study_mediation_{FRAMING_SOURCE}_{FRAMING_TAG}.csv'
mediation.to_csv(mediation_path, index=False)
print(f'Saved: {mediation_path}')
mediation


## Visualizations

In [ ]:
def scatter_with_fit(ax, df, x_col, y_col, title, xlabel, ylabel):
    clean = df[[x_col, y_col, 'Country']].dropna()
    ax.scatter(clean[x_col], clean[y_col], color='#1f77b4', s=60)
    slope, intercept, r, p, _ = stats.linregress(clean[x_col], clean[y_col])
    xs = np.linspace(clean[x_col].min(), clean[x_col].max(), 100)
    ax.plot(xs, intercept + slope * xs, color='#444', linewidth=1.5)
    for _, row in clean.iterrows():
        ax.annotate(row['Country'], (row[x_col], row[y_col]), xytext=(4, 4), textcoords='offset points', fontsize=8)
    title_text = title + chr(10) + f'r={r:.2f}, p={p:.3f}'
    ax.set_title(title_text, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)


fig, axes = plt.subplots(2, 2, figsize=(13, 10))
scatter_with_fit(
    axes[0, 0], study,
    'AI Vibrancy Score', 'Innovation Share',
    'H1: Vibrancy vs innovation framing',
    'Stanford AI Vibrancy Score', 'Innovation-oriented sentence share',
)
scatter_with_fit(
    axes[0, 1], study,
    'AI Vibrancy Score', 'Risk Share',
    'H2: Vibrancy vs risk framing',
    'Stanford AI Vibrancy Score', 'Risk-oriented sentence share',
)
scatter_with_fit(
    axes[1, 0], study,
    'IEF Overall Score', 'Innovation Share',
    'H3: Economic freedom vs innovation framing',
    'IEF Overall Score', 'Innovation-oriented sentence share',
)
scatter_with_fit(
    axes[1, 1], study,
    'Risk Share', 'IEF Overall Score',
    'H4: Risk framing vs economic freedom',
    'Risk-oriented sentence share', 'IEF Overall Score',
)

fig.suptitle(f'Correlation Study ({FRAMING_SOURCE}, {FRAMING_TAG})', fontsize=14, fontweight='bold')
fig.tight_layout()
figure_path = outputs / f'correlation_study_scatter_{FRAMING_SOURCE}_{FRAMING_TAG}.png'
fig.savefig(figure_path, dpi=200, bbox_inches='tight')
print(f'Saved: {figure_path}')
plt.show()


In [ ]:
matrix_cols = [
    'AI Vibrancy Score',
    'IEF Overall Score',
    'Innovation Share',
    'Risk Share',
    'Mean Framing Score',
    'Innovation to Risk Ratio',
    'Log Innovation to Risk Ratio',
]
corr_matrix = study[matrix_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(8.5, 6.5))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(matrix_cols)))
ax.set_xticklabels(matrix_cols, rotation=45, ha='right')
ax.set_yticks(range(len(matrix_cols)))
ax.set_yticklabels(matrix_cols)
for i in range(len(matrix_cols)):
    for j in range(len(matrix_cols)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title(f'Pearson Correlation Matrix ({FRAMING_SOURCE}, {FRAMING_TAG})')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
matrix_path = outputs / f'correlation_study_matrix_{FRAMING_SOURCE}_{FRAMING_TAG}.png'
fig.savefig(matrix_path, dpi=200, bbox_inches='tight')
print(f'Saved: {matrix_path}')
plt.show()


## Interpretation Template

Use the coefficient direction and correlation strength rather than p-values alone, because the sample is only eight countries.

- H1 is supported directionally if AI Vibrancy is positively correlated with `Innovation Share`.
- H2 is supported directionally if AI Vibrancy is negatively correlated with `Risk Share`, meaning lower-vibrancy countries tend to use more risk-oriented language.
- H3 is supported directionally if IEF is positively correlated with `Innovation Share`.
- H4 is supported directionally if `Risk Share` is negatively correlated with IEF, or equivalently if more risk-oriented rhetoric is associated with lower economic freedom.
- The log innovation-to-risk ratio and mean framing score are robustness measures for the same underlying innovation-versus-risk balance.
